# 🧠 Txoko — Chatbot turístico de Euskadi con RAG
### Documentación técnica del sistema

Este notebook explica paso a paso cómo hemos construido **Txoko**, un chatbot de recomendaciones turísticas para Euskadi basado en la arquitectura **RAG (Retrieval Augmented Generation)**.

---

## ¿Qué es RAG y por qué lo usamos?

Un LLM (modelo de lenguaje grande) como Gemini sabe muchas cosas de forma general, pero **no conoce datos específicos** de tu negocio, proyecto o dataset. RAG resuelve esto:

1. Buscas en tu propia base de datos los documentos más relevantes para la pregunta del usuario
2. Se los pasas al LLM como contexto
3. El LLM responde basándose en esos datos reales, no en suposiciones

```
Usuario → pregunta
   ↓
Búsqueda vectorial (FAISS) → recupera lugares relevantes
   ↓
Construcción de prompt con esos lugares
   ↓
LLM (Gemini) → responde con información real
   ↓
Usuario ← respuesta fundamentada
```

---

## Estructura del proyecto

```
/rag/                              ← carpeta raíz
├── venv/                          ← entorno virtual Python
├── build_faiss_index.py           ← script de construcción del índice
├── test_faiss.py                  ← script de pruebas de retrieval
├── aupa_master_v5.csv             ← datos de lugares turísticos
├── txoko_reviews_raw.json         ← reseñas de usuarios
├── rag_assets/
│   ├── faiss_index.index          ← índice vectorial serializado
│   └── faiss_metadata.json        ← metadatos sincronizados con el índice
└── backend/
    ├── main.py                    ← servidor FastAPI
    ├── requirements.txt
    ├── .env.example
    └── rag/                       ← módulo Python del backend
        ├── __init__.py
        ├── config.py
        ├── index_loader.py
        ├── retrieval.py
        ├── prompt.py
        └── llm.py
```

---

# FASE 1 — Construcción del índice vectorial

## ¿Qué es un embedding?

Un **embedding** es la representación numérica de un texto. Un modelo de lenguaje convierte cualquier frase en un vector de números (por ejemplo, 384 números). La propiedad clave es que **textos semánticamente similares producen vectores cercanos en el espacio vectorial**.

Ejemplo:
- "pintxos en Bilbao" → [0.23, -0.41, 0.87, ...]
- "bar de tapas vasco" → [0.25, -0.39, 0.85, ...]  ← muy cercano
- "museo de arte moderno" → [-0.12, 0.67, -0.34, ...] ← lejano

## Modelo elegido: `paraphrase-multilingual-MiniLM-L12-v2`

Este modelo de [sentence-transformers](https://www.sbert.net/) fue elegido porque:
- Es **multilingüe** (funciona en castellano y euskera)
- Es **ligero** (funciona bien sin GPU)
- Produce vectores de **384 dimensiones**
- Está optimizado para **similitud semántica**

In [ ]:
# ── Instalación de dependencias ────────────────────────────────────────────
# Ejecutar solo si no están instaladas en el entorno

# !pip install sentence-transformers faiss-cpu pandas numpy google-genai fastapi uvicorn

In [ ]:
# ── Carga del modelo de embeddings ────────────────────────────────────────
from sentence_transformers import SentenceTransformer
import numpy as np

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)

print(f"Modelo cargado: {MODEL_NAME}")
print(f"Dimensión de los embeddings: {model.get_sentence_embedding_dimension()}")

In [ ]:
# ── Ejemplo: qué produce un embedding ─────────────────────────────────────
textos = [
    "pintxos en Bilbao",
    "bar de tapas vasco",
    "museo de arte moderno",
]

embeddings = model.encode(textos)

print(f"Shape de los embeddings: {embeddings.shape}")
print(f"\nPrimeros 5 valores del embedding de '{textos[0]}':")
print(embeddings[0][:5])

In [ ]:
# ── Verificación de similitud semántica ───────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity

sim_01 = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
sim_02 = cosine_similarity([embeddings[0]], [embeddings[2]])[0][0]

print(f"Similitud '{textos[0]}' vs '{textos[1]}': {sim_01:.3f}")
print(f"Similitud '{textos[0]}' vs '{textos[2]}': {sim_02:.3f}")
print("\n→ Mayor similitud = textos semánticamente más cercanos")

## Construcción del índice FAISS

**FAISS** (Facebook AI Similarity Search) es una librería que permite buscar de forma eficiente los vectores más cercanos a uno dado entre millones de vectores. Es el motor de búsqueda vectorial del sistema.

El proceso es:
1. Cargar los datos de lugares turísticos (CSV) y reseñas (JSON)
2. Construir un texto enriquecido por cada lugar
3. Generar el embedding de cada texto
4. Indexar todos los embeddings en FAISS
5. Guardar el índice y los metadatos en disco

In [ ]:
# ── Carga de datos ─────────────────────────────────────────────────────────
import pandas as pd
import json
from pathlib import Path

BASE_DIR = Path(".")  # ajustar a la ruta raíz del proyecto si es necesario

df = pd.read_csv(BASE_DIR / "aupa_master_v5.csv")

with open(BASE_DIR / "txoko_reviews_raw.json", "r", encoding="utf-8") as f:
    reviews_raw = json.load(f)

print(f"Lugares cargados: {len(df)}")
print(f"Columnas disponibles: {list(df.columns)}")

In [ ]:
# ── Construcción de textos enriquecidos ────────────────────────────────────
# Para cada lugar construimos un texto que concatena todos sus campos relevantes.
# Esto permite que el embedding capture toda la información semántica del lugar.

def build_text(row: pd.Series, reviews: dict) -> str:
    parts = [
        str(row.get("nombre", "")),
        str(row.get("categoria", "") or row.get("categoría", "")),
        str(row.get("municipio", "")),
        str(row.get("provincia", "")),
        str(row.get("descripcion", "") or row.get("descripción", "")),
    ]
    # Añadir reseñas si existen para este lugar
    lugar_id = str(row.get("id", ""))
    if lugar_id in reviews:
        reseñas_texto = " ".join(reviews[lugar_id][:3])  # máximo 3 reseñas
        parts.append(reseñas_texto)

    return " | ".join(p for p in parts if p and p != "nan")

# Ejemplo con el primer lugar
texto_ejemplo = build_text(df.iloc[0], reviews_raw)
print("Texto enriquecido de ejemplo:")
print(texto_ejemplo)

In [ ]:
# ── Generación de embeddings para todos los lugares ────────────────────────
import faiss

textos_lugares = [build_text(row, reviews_raw) for _, row in df.iterrows()]

print(f"Generando embeddings para {len(textos_lugares)} lugares...")
embeddings_lugares = model.encode(
    textos_lugares,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype("float32")

print(f"\nShape del array de embeddings: {embeddings_lugares.shape}")
print("→ Cada fila es un lugar, cada columna una dimensión del embedding")

In [ ]:
# ── Construcción del índice FAISS ──────────────────────────────────────────
# Usamos IndexFlatL2: búsqueda exacta por distancia euclídea.
# Para datasets grandes se usaría IndexIVFFlat (aproximado, más rápido).

dimension = embeddings_lugares.shape[1]  # 384
index = faiss.IndexFlatL2(dimension)
index.add(embeddings_lugares)

print(f"Índice FAISS construido")
print(f"Vectores indexados: {index.ntotal}")
print(f"Dimensión: {dimension}")

In [ ]:
# ── Guardado del índice y metadatos ───────────────────────────────────────
# Los metadatos son la información original de cada lugar,
# sincronizados por índice con los vectores FAISS.

RAG_ASSETS_DIR = BASE_DIR / "rag_assets"
RAG_ASSETS_DIR.mkdir(exist_ok=True)

# Guardar índice FAISS
faiss.write_index(index, str(RAG_ASSETS_DIR / "faiss_index.index"))

# Guardar metadatos como JSON
metadata = df.to_dict(orient="records")
with open(RAG_ASSETS_DIR / "faiss_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("✅ Índice y metadatos guardados en rag_assets/")

---

# FASE 2 — Backend FastAPI

## ¿Qué es FastAPI?

**FastAPI** es un framework Python para construir APIs web de forma rápida. En nuestro caso, es el servidor que recibe las preguntas del usuario y orquesta todo el pipeline RAG.

Ventajas clave para este proyecto:
- Carga el modelo y el índice **una sola vez** al arrancar (no por cada petición)
- Genera documentación interactiva automáticamente en `/docs`
- Tipado con Pydantic para validar las peticiones

## Arquitectura modular del backend

El backend se divide en módulos con responsabilidades claras:

| Archivo | Responsabilidad |
|---|---|
| `config.py` | Todas las variables configurables (rutas, modelos, parámetros) |
| `index_loader.py` | Carga el índice FAISS y metadatos desde disco |
| `retrieval.py` | Encode de la query + búsqueda FAISS + filtros |
| `prompt.py` | Construcción del prompt para el LLM |
| `llm.py` | Llamada a la API de Gemini |
| `main.py` | Servidor FastAPI: orquesta todo lo anterior |

## Código completo del backend

A continuación se muestra el código de cada módulo tal y como está implementado.

In [ ]:
# ── backend/rag/config.py ─────────────────────────────────────────────────
# Todas las variables configurables en un único lugar.
# Para cambiar el modelo, los parámetros de retrieval o las rutas,
# solo hay que tocar este archivo.

CONFIG_CODE = '''
import os
from pathlib import Path

# Rutas
BASE_DIR = Path(__file__).resolve().parent.parent.parent  # /rag/
RAG_DIR  = BASE_DIR / "rag_assets"

FAISS_INDEX_PATH    = RAG_DIR / "faiss_index.index"
FAISS_METADATA_PATH = RAG_DIR / "faiss_metadata.json"

# Embedding model
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Retrieval
FAISS_TOP_K = 10   # candidatos recuperados de FAISS
FINAL_TOP_K = 5    # lugares enviados al LLM

# LLM
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
GEMINI_MODEL   = "gemini-2.0-flash"

# System prompt
SYSTEM_PROMPT = (
    "Eres un asistente de recomendaciones turísticas de Euskadi llamado Txoko.\\n"
    "Responde siempre en el idioma del usuario.\\n"
    "Usa únicamente la información proporcionada en el contexto.\\n"
    "No inventes datos como horarios, precios ni distancias.\\n"
    "Si no tienes suficiente información, dilo claramente.\\n"
    "Sé claro, útil y ligeramente cercano en tono."
)
'''
print(CONFIG_CODE)

In [ ]:
# ── backend/rag/index_loader.py ───────────────────────────────────────────
# Carga el índice FAISS y los metadatos desde disco.
# Se llama UNA SOLA VEZ al arrancar el servidor.

INDEX_LOADER_CODE = '''
import json
import faiss
from pathlib import Path
from .config import FAISS_INDEX_PATH, FAISS_METADATA_PATH

def load_faiss_index(index_path=FAISS_INDEX_PATH):
    if not index_path.exists():
        raise FileNotFoundError(f"FAISS index not found: {index_path}")
    return faiss.read_index(str(index_path))

def load_metadata(metadata_path=FAISS_METADATA_PATH):
    if not metadata_path.exists():
        raise FileNotFoundError(f"Metadata file not found: {metadata_path}")
    with open(metadata_path, "r", encoding="utf-8") as f:
        return json.load(f)
'''
print(INDEX_LOADER_CODE)

In [ ]:
# ── backend/rag/retrieval.py ──────────────────────────────────────────────
# Pipeline: encode query → búsqueda FAISS → filtro post-retrieval → top-k
#
# FILTRO POST-RETRIEVAL (MVP):
# Si la query menciona un municipio conocido de Euskadi, se filtran
# los candidatos para devolver solo los de ese municipio.
# Si el filtro deja 0 resultados (p.ej. el municipio está escrito en euskera
# pero los datos están en castellano), se devuelven los top-k sin filtrar
# para no degradar la respuesta.

RETRIEVAL_CODE = '''
import re
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from .config import FAISS_TOP_K, FINAL_TOP_K

def encode_query(query, model):
    vector = model.encode([query], convert_to_numpy=True)
    return vector.astype("float32")

def search_faiss(query_vector, index, metadata, k=FAISS_TOP_K):
    distances, indices = index.search(query_vector, k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue
        entry = metadata[idx].copy()
        entry["_score"] = float(dist)
        results.append(entry)
    return results

def apply_filters(candidates, query):
    MUNICIPIOS = [
        "bilbao", "donostia", "san sebastián", "vitoria", "gasteiz",
        "barakaldo", "getxo", "irun", "eibar", "zarautz",
        # ... ampliar según necesidades
    ]
    query_lower = query.lower()
    municipio = next(
        (m for m in MUNICIPIOS if re.search(rf"\\b{re.escape(m)}\\b", query_lower)),
        None
    )
    if municipio:
        filtered = [
            c for c in candidates
            if municipio in (c.get("municipio") or "").lower()
        ]
        if len(filtered) >= 1:
            return filtered[:FINAL_TOP_K]
    return candidates[:FINAL_TOP_K]

def retrieve(query, model, index, metadata):
    vector     = encode_query(query, model)
    candidates = search_faiss(vector, index, metadata)
    return apply_filters(candidates, query)
'''
print(RETRIEVAL_CODE)

In [ ]:
# ── backend/rag/prompt.py ─────────────────────────────────────────────────
# Construye el prompt que se envía al LLM.
#
# El prompt tiene dos partes:
#   1. System prompt fijo: define el rol y las restricciones del asistente
#   2. User section dinámica: la pregunta del usuario + los lugares recuperados
#
# Gemini Flash acepta ambas partes en un único string de texto.

PROMPT_CODE = '''
from .config import SYSTEM_PROMPT

def _format_place(i, place):
    nombre      = place.get("nombre", "Sin nombre")
    municipio   = place.get("municipio", "")
    provincia   = place.get("provincia", "")
    categoria   = place.get("categoria") or place.get("categoría", "")
    descripcion = place.get("descripcion") or place.get("descripción", "")
    rating      = place.get("google_rating")
    num_reviews = place.get("google_num_reviews")

    ubicacion = ", ".join(filter(None, [municipio, provincia]))
    lines = [f"{i}. {nombre} — {ubicacion} ({categoria})"]
    lines.append(f"   Descripción: {descripcion}")
    if rating is not None:
        review_str = f" ({int(num_reviews)} reseñas)" if num_reviews else ""
        lines.append(f"   Rating: {rating}{review_str}")
    return "\\n".join(lines)

def build_prompt(query, places):
    places_block = "\\n\\n".join(
        _format_place(i + 1, p) for i, p in enumerate(places)
    )
    user_section = (
        f\'El usuario pregunta: "{query}"\\n\\n\'
        f"Lugares disponibles:\\n\\n"
        f"{places_block}"
    )
    return f"{SYSTEM_PROMPT}\\n\\n---\\n\\n{user_section}"
'''
print(PROMPT_CODE)

In [ ]:
# ── backend/rag/llm.py ────────────────────────────────────────────────────
# Integración con Gemini API usando la librería oficial google-genai.
#
# NOTA: La librería anterior google-generativeai está deprecada.
# Usar google-genai (pip install google-genai).

LLM_CODE = '''
import google.genai as genai
from .config import GEMINI_API_KEY, GEMINI_MODEL

_client = None

def init_gemini():
    global _client
    if not GEMINI_API_KEY:
        raise EnvironmentError(
            "GEMINI_API_KEY no encontrada. "
            "Define la variable de entorno antes de iniciar el servidor."
        )
    _client = genai.Client(api_key=GEMINI_API_KEY)

def call_llm(prompt):
    response = _client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
    )
    return response.text
'''
print(LLM_CODE)

In [ ]:
# ── backend/main.py ───────────────────────────────────────────────────────
# Servidor FastAPI principal.
#
# Puntos clave:
#   - lifespan: carga modelo, índice y Gemini UNA SOLA VEZ al arrancar
#   - AppState: almacena en memoria los objetos pesados
#   - GET /health: endpoint de comprobación
#   - POST /chat: endpoint principal del chatbot

MAIN_CODE = '''
from contextlib import asynccontextmanager
import faiss
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer

from rag.config import EMBEDDING_MODEL
from rag.index_loader import load_faiss_index, load_metadata
from rag.llm import call_llm, init_gemini
from rag.prompt import build_prompt
from rag.retrieval import retrieve

class AppState:
    index:    faiss.Index
    metadata: list[dict]
    model:    SentenceTransformer

state = AppState()

@asynccontextmanager
async def lifespan(app):
    print("⏳ Cargando modelo de embeddings...")
    state.model = SentenceTransformer(EMBEDDING_MODEL)
    print("⏳ Cargando índice FAISS y metadatos...")
    state.index    = load_faiss_index()
    state.metadata = load_metadata()
    print("⏳ Inicializando Gemini...")
    init_gemini()
    print("✅ Txoko backend listo.")
    yield

app = FastAPI(title="Txoko", version="0.1.0", lifespan=lifespan)

class ChatRequest(BaseModel):
    query: str

class ChatResponse(BaseModel):
    answer: str

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest):
    query = request.query.strip()
    if not query:
        raise HTTPException(status_code=422, detail="La query no puede estar vacía.")
    places = retrieve(query, state.model, state.index, state.metadata)
    if not places:
        return ChatResponse(answer="No he encontrado lugares relevantes.")
    prompt = build_prompt(query, places)
    try:
        answer = call_llm(prompt)
    except Exception as e:
        raise HTTPException(status_code=502, detail=f"Error al llamar al LLM: {e}")
    return ChatResponse(answer=answer)
'''
print(MAIN_CODE)

---

# FASE 3 — Puesta en marcha

## Dependencias necesarias

Todo el sistema requiere las siguientes librerías instaladas en el entorno virtual:

| Librería | Para qué se usa |
|---|---|
| `sentence-transformers` | Modelo de embeddings multilingüe |
| `faiss-cpu` | Índice vectorial de búsqueda |
| `google-genai` | API de Gemini (LLM) |
| `fastapi` | Framework del servidor web |
| `uvicorn` | Servidor ASGI para correr FastAPI |
| `pandas` | Carga y manipulación del CSV |
| `numpy` | Operaciones con arrays de embeddings |

In [ ]:
# ── Arranque del servidor ─────────────────────────────────────────────────
# Ejecutar estos comandos en la terminal, NO en el notebook.

ARRANQUE = """
# 1. Activar el entorno virtual
source /rag/venv/bin/activate          # Linux / Mac
# rag\\venv\\Scripts\\activate.bat      # Windows CMD
# rag/venv/Scripts/Activate.ps1        # Windows PowerShell

# 2. Definir la API key de Gemini como variable de entorno
export GEMINI_API_KEY=tu_api_key_aqui  # Linux / Mac
# set GEMINI_API_KEY=tu_api_key_aqui   # Windows CMD
# $env:GEMINI_API_KEY="tu_api_key"     # Windows PowerShell

# 3. Arrancar el servidor desde la carpeta backend/
cd /rag/backend
uvicorn main:app --reload
"""

print(ARRANQUE)

## Verificación del sistema

Una vez el servidor está corriendo, verás en la terminal:

```
⏳ Cargando modelo de embeddings...
⏳ Cargando índice FAISS y metadatos...
⏳ Inicializando Gemini...
✅ Txoko backend listo.
INFO:     Uvicorn running on http://127.0.0.1:8000
```

Tienes dos formas de probar el chatbot:

**Opción A — Interfaz interactiva (recomendada):**
Abre `http://127.0.0.1:8000/docs` en el navegador → POST /chat → Try it out

**Opción B — curl desde terminal:**
```bash
curl -X POST http://127.0.0.1:8000/chat \
  -H "Content-Type: application/json" \
  -d "{\"query\": \"restaurante de pintxos en Bilbao\"}"
```

**Respuesta esperada:**
```json
{
  "answer": "En Bilbao tienes varias opciones para disfrutar de pintxos..."
}
```

---

## Resumen del flujo completo

```
POST /chat  {"query": "pintxos cerca del Guggenheim"}
      │
      ▼
retrieval.py  →  encode_query()  →  vector float32 [1, 384]
      │
      ▼
retrieval.py  →  search_faiss()  →  10 candidatos con _score
      │
      ▼
retrieval.py  →  apply_filters()  →  5 lugares finales (filtro municipio)
      │
      ▼
prompt.py    →  build_prompt()  →  system prompt + lugares formateados
      │
      ▼
llm.py       →  call_llm()  →  Gemini API  →  texto de respuesta
      │
      ▼
POST /chat  {"answer": "Te recomiendo..."}
```

---

## Posibles extensiones futuras

El sistema está diseñado para escalar. Algunas extensiones naturales:

- **Búsqueda híbrida:** combinar FAISS (semántica) con búsqueda por keywords (BM25)
- **Reranking:** usar un cross-encoder para reordenar los candidatos antes de enviarlos al LLM
- **Cache:** almacenar respuestas de queries frecuentes para reducir latencia y coste
- **Streaming:** devolver la respuesta del LLM token a token para mejor UX
- **Filtros avanzados:** por categoría, rating mínimo, o coordenadas GPS